In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score
import glob
import os
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

def load_and_invert(file_path, type_model = 'single'):
    
    y_true, y_prob = load_predictions(file_path, type_model = type_model)
    y_true = 1 - y_true    # invert class labels
    y_prob = 1 - y_prob    # invert probabilities
    print(type_model, len(y_true))
    return y_true, y_prob

def load_predictions(file_path, type_model='single'):
    df = pd.read_csv(file_path)
    
    # Handle ensemble files (y_true, y_prob_pos)
    if 'y_true' in df.columns and 'y_prob_pos' in df.columns:
        y_true = df['y_true'].values
        y_prob = df['y_prob_pos'].values
    
    # Handle individual model files (true_label, prob_class_1)
    elif 'true_label' in df.columns and 'prob_class_1' in df.columns:
        y_true = df['true_label'].values
        y_prob = df['prob_class_1'].values
    
    else:
        raise ValueError(f"Unknown column format in {file_path}. Columns: {df.columns.tolist()}")
    
    return y_true, y_prob


def compute_metrics(file_path, threshold=0.5, type_model='single'):
    """Compute all metrics for inverted classes."""
    y_true, y_prob = load_and_invert(file_path, type_model=type_model)
    
    # Apply threshold
    y_pred = (y_prob >= threshold).astype(int)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Metrics
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    balanced_accuracy = (sensitivity + specificity) / 2
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    auc = roc_auc_score(y_true, y_prob)
    
    return {
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Balanced_Accuracy': balanced_accuracy,
        'F1': f1,
        'AUC': auc,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn
    }


# ===========================================================
# 1. GET ALL INDIVIDUAL CALIBRATION MODEL FILES
# ===========================================================
individual_cal_files = sorted(
    glob.glob("/projects/retprogression/rgarridogarcia/test_split/predictions_model_*_calibration.csv")
)

print("Found individual calibration models:", len(individual_cal_files))


# ===========================================================
# 2. GET ALL INDIVIDUAL VALIDATION MODEL FILES
# ===========================================================
individual_val_files = sorted(
    glob.glob("/projects/retprogression/rgarridogarcia/test_split/predictions_model_*_validation.csv")
)

print("Found individual validation models:", len(individual_val_files))


# ===========================================================
# 3. GET CALIBRATION ENSEMBLE FILES
# ===========================================================
ensemble_cal_dir = "/projects/retprogression/rgarridogarcia/ensemble/ensemble_calibration_half_split"

all_ensemble_cal_files = glob.glob(os.path.join(ensemble_cal_dir, "ensemble_*.csv"))

ensemble_cal_files = []
for f in all_ensemble_cal_files:
    df = pd.read_csv(f, nrows=1)
    if ("y_true" in df.columns) and ("y_prob_pos" in df.columns):
        ensemble_cal_files.append(f)

print("Valid ensemble calibration files:", len(ensemble_cal_files))


# ===========================================================
# 4. GET VALIDATION ENSEMBLE FILES
# ===========================================================
ensemble_val_dir = "/projects/retprogression/rgarridogarcia/ensemble/ensemble_validation_half_split"

all_ensemble_val_files = glob.glob(os.path.join(ensemble_val_dir, "ensemble_*.csv"))

ensemble_val_files = []
for f in all_ensemble_val_files:
    df = pd.read_csv(f, nrows=1)
    if ("y_true" in df.columns) and ("y_prob_pos" in df.columns):
        ensemble_val_files.append(f)

print("Valid ensemble validation files:", len(ensemble_val_files))


def load_predictions_with_ids(file_path, type_model='single'):
    """Load predictions along with image IDs from CSV."""
    df = pd.read_csv(file_path)

    if 'y_true' in df.columns and 'y_prob_pos' in df.columns:
        image_ids = df['id'].values
        y_true = df['y_true'].values
        y_prob = df['y_prob_pos'].values
    elif 'true_label' in df.columns and 'prob_class_1' in df.columns:
        image_ids = df['image_id'].values
        y_true = df['true_label'].values
        y_prob = df['prob_class_1'].values
    else:
        raise ValueError(f"Unknown column format in {file_path}. Columns: {df.columns.tolist()}")

    # Invert classes (same as load_and_invert)
    y_true = 1 - y_true
    y_prob = 1 - y_prob

    return image_ids, y_true, y_prob


def get_confusion_ids(file_path, threshold=0.5, type_model='single'):
    """Return dict with TP/FP/FN/TN image ID lists for a given file."""
    image_ids, y_true, y_prob = load_predictions_with_ids(file_path, type_model)
    y_pred = (y_prob >= threshold).astype(int)

    tp_ids = image_ids[(y_true == 1) & (y_pred == 1)].tolist()
    fp_ids = image_ids[(y_true == 0) & (y_pred == 1)].tolist()
    fn_ids = image_ids[(y_true == 1) & (y_pred == 0)].tolist()
    tn_ids = image_ids[(y_true == 0) & (y_pred == 0)].tolist()

    return {'TP': tp_ids, 'FP': fp_ids, 'FN': fn_ids, 'TN': tn_ids}


# ===========================================================
# BUILD DICTIONARIES FOR CALIBRATION (test1)
# ===========================================================
TP_test1, FP_test1, FN_test1, TN_test1 = {}, {}, {}, {}

# Individual models
for i in range(1, 6):
    file = [f for f in individual_cal_files if f"predictions_model_{i}_" in f][0]
    ids = get_confusion_ids(file, threshold=0.5, type_model='single')
    model_name = f'Model {i}'
    TP_test1[model_name] = ids['TP']
    FP_test1[model_name] = ids['FP']
    FN_test1[model_name] = ids['FN']
    TN_test1[model_name] = ids['TN']

# Ensemble methods
for f in ensemble_cal_files:
    method_name = os.path.basename(f).replace('ensemble_', '').replace('.csv', '').replace('_', ' ').title()
    ids = get_confusion_ids(f, threshold=0.5, type_model='ensemble')
    TP_test1[method_name] = ids['TP']
    FP_test1[method_name] = ids['FP']
    FN_test1[method_name] = ids['FN']
    TN_test1[method_name] = ids['TN']


# ===========================================================
# BUILD DICTIONARIES FOR VALIDATION (test2)
# ===========================================================
TP_test2, FP_test2, FN_test2, TN_test2 = {}, {}, {}, {}

# Individual models
for i in range(1, 6):
    file = [f for f in individual_val_files if f"predictions_model_{i}_" in f][0]
    ids = get_confusion_ids(file, threshold=0.5, type_model='single')
    model_name = f'Model {i}'
    TP_test2[model_name] = ids['TP']
    FP_test2[model_name] = ids['FP']
    FN_test2[model_name] = ids['FN']
    TN_test2[model_name] = ids['TN']

# Ensemble methods
for f in ensemble_val_files:
    method_name = os.path.basename(f).replace('ensemble_', '').replace('.csv', '').replace('_', ' ').title()
    ids = get_confusion_ids(f, threshold=0.5, type_model='ensemble')
    TP_test2[method_name] = ids['TP']
    FP_test2[method_name] = ids['FP']
    FN_test2[method_name] = ids['FN']
    TN_test2[method_name] = ids['TN']


# ===========================================================
# QUICK SANITY CHECK
# ===========================================================
print("Calibration (test1) — keys:", list(TP_test1.keys()))
print("Validation  (test2) — keys:", list(TP_test2.keys()))

# Spot-check counts match the metrics table
for name in TP_test1:
    n = len(TP_test1[name]) + len(FP_test1[name]) + len(FN_test1[name]) + len(TN_test1[name])
    print(f"  [test1] {name}: TP={len(TP_test1[name])}, FP={len(FP_test1[name])}, FN={len(FN_test1[name])}, TN={len(TN_test1[name])}, total={n}")

Found individual calibration models: 5
Found individual validation models: 5
Valid ensemble calibration files: 8
Valid ensemble validation files: 8
Calibration (test1) — keys: ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5', 'Average', 'Confidence', 'Logit', 'Majority 60', 'Majority 80', 'Max Confidence', 'Median', 'Unanimous']
Validation  (test2) — keys: ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5', 'Average', 'Confidence', 'Logit', 'Majority 60', 'Majority 80', 'Max Confidence', 'Median', 'Unanimous']
  [test1] Model 1: TP=11, FP=10, FN=8, TN=796, total=825
  [test1] Model 2: TP=14, FP=29, FN=5, TN=777, total=825
  [test1] Model 3: TP=13, FP=23, FN=6, TN=783, total=825
  [test1] Model 4: TP=11, FP=22, FN=8, TN=784, total=825
  [test1] Model 5: TP=14, FP=19, FN=5, TN=787, total=825
  [test1] Average: TP=14, FP=18, FN=5, TN=788, total=825
  [test1] Confidence: TP=14, FP=19, FN=5, TN=787, total=825
  [test1] Logit: TP=14, FP=18, FN=5, TN=788, total=825
  [test1] Majorit

In [12]:
# Spot-check counts match the metrics table
for name in TP_test1:
    n = len(TP_test1[name]) + len(FP_test1[name]) + len(FN_test1[name]) + len(TN_test1[name])
    print(f"  [test1] {name}: FN={FN_test1[name]}")

  [test1] Model 1: FN=[605367, 617557, 625049, 625052, 625154, 626306, 629433, 636595]
  [test1] Model 2: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Model 3: FN=[605367, 617557, 626306, 629433, 636595, 643545]
  [test1] Model 4: FN=[605367, 617557, 625049, 625154, 626306, 629433, 636595, 643545]
  [test1] Model 5: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Average: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Confidence: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Logit: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Majority 60: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Majority 80: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Max Confidence: FN=[605367, 617557, 625049, 625154, 626306, 629433, 636595]
  [test1] Median: FN=[605367, 617557, 626306, 629433, 636595]
  [test1] Unanimous: FN=[605367, 617557, 626306, 629433, 636595]


### FN

In [13]:
# ===========================================================
# FN IMAGES MISSED BY ALL MODELS (calibration & validation)
# ===========================================================

# --- Calibration (test1) ---
fn_sets_test1 = [set(ids) for ids in FN_test1.values()]
fn_all_test1  = set.intersection(*fn_sets_test1)
fn_unique_test1 = sorted(set.union(*fn_sets_test1))

print(f"Calibration (test1): {len(fn_all_test1)} images are FN in ALL models")
print(f"Unique FN image IDs (any model): {fn_unique_test1}")
print(fn_all_test1)

# --- Validation (test2) ---
fn_sets_test2 = [set(ids) for ids in FN_test2.values()]
fn_all_test2  = set.intersection(*fn_sets_test2)
fn_unique_test2 = sorted(set.union(*fn_sets_test2))

print(f"\nValidation  (test2): {len(fn_all_test2)} images are FN in ALL models")
print(f"Unique FN image IDs (any model): {fn_unique_test2}")
print(fn_all_test2)

Calibration (test1): 5 images are FN in ALL models
Unique FN image IDs (any model): [605367, 617557, 625049, 625052, 625154, 626306, 629433, 636595, 643545]
{626306, 636595, 617557, 605367, 629433}

Validation  (test2): 6 images are FN in ALL models
Unique FN image IDs (any model): [605368, 618210, 625051, 626305, 629432, 635204, 635205, 636596]
{626305, 635204, 635205, 636596, 629432, 605368}


In [14]:
import shutil

data_dir   = "/projects/retprogression/test_dataset_10032025"
out_test1  = "/projects/retprogression/Gradable_allWrong_FN_test1"
out_test2  = "/projects/retprogression/Gradable_allWrong_FN_test2"

os.makedirs(out_test1, exist_ok=True)
os.makedirs(out_test2, exist_ok=True)

def copy_images(image_ids, src_dir, dst_dir):
    copied, missing = 0, []
    for img_id in image_ids:
        # Glob in case the extension varies (.jpg, .png, etc.)
        matches = glob.glob(os.path.join(src_dir, f"{img_id}.*"))
        if matches:
            for src in matches:
                shutil.copy2(src, dst_dir)
            copied += 1
        else:
            missing.append(img_id)
    return copied, missing

# --- Calibration FNs ---
copied1, missing1 = copy_images(fn_all_test1, data_dir, out_test1)
print(f"[test1] Copied: {copied1} | Not found: {len(missing1)}")
if missing1:
    print(f"  Missing IDs: {missing1}")

# --- Validation FNs ---
copied2, missing2 = copy_images(fn_all_test2, data_dir, out_test2)
print(f"[test2] Copied: {copied2} | Not found: {len(missing2)}")
if missing2:
    print(f"  Missing IDs: {missing2}")

[test1] Copied: 5 | Not found: 0
[test2] Copied: 6 | Not found: 0


### FP

In [15]:
# ===========================================================
# FN IMAGES MISSED BY ALL MODELS (calibration & validation)
# ===========================================================

# --- Calibration (test1) ---
fp_sets_test1 = [set(ids) for ids in FP_test1.values()]
fp_all_test1  = set.intersection(*fp_sets_test1)
fp_unique_test1 = sorted(set.union(*fp_sets_test1))

print(f"Calibration (test1): {len(fp_all_test1)} images are FP in ALL models")
print(f"Unique FP image IDs (any model): {fp_unique_test1}")
print(fp_all_test1)

# --- Validation (test2) ---
fp_sets_test2 = [set(ids) for ids in FP_test2.values()]
fp_all_test2  = set.intersection(*fp_sets_test2)
fp_unique_test2 = sorted(set.union(*fp_sets_test2))

print(f"\nValidation  (test2): {len(fp_all_test2)} images are FP in ALL models")
print(f"Unique FP image IDs (any model): {fp_unique_test2}")
print(fp_all_test2)

Calibration (test1): 10 images are FP in ALL models
Unique FP image IDs (any model): [606504, 608148, 608716, 610297, 610630, 611220, 612532, 613915, 617114, 617274, 618800, 620075, 620652, 620653, 622857, 624981, 624988, 627008, 629132, 629730, 629733, 629995, 631545, 631862, 631867, 631868, 632064, 637465, 638490, 639203, 639205, 644699, 645189, 651177, 651179]
{627008, 639203, 629733, 651177, 629995, 629132, 651179, 610297, 644699, 631867}

Validation  (test2): 6 images are FP in ALL models
Unique FP image IDs (any model): [605631, 610296, 611219, 620077, 622859, 629133, 631364, 631366, 631546, 631869, 632063, 632065, 632066, 632508, 632509, 634828, 634829, 639204, 640590, 644700, 651178, 651180]
{631364, 631366, 651178, 651180, 610296, 644700}


In [16]:
import shutil

data_dir   = "/projects/retprogression/test_dataset_10032025"
out_test1  = "/projects/retprogression/Gradable_allWrong_FP_test1"
out_test2  = "/projects/retprogression/Gradable_allWrong_FP_test2"

os.makedirs(out_test1, exist_ok=True)
os.makedirs(out_test2, exist_ok=True)

def copy_images(image_ids, src_dir, dst_dir):
    copied, missing = 0, []
    for img_id in image_ids:
        # Glob in case the extension varies (.jpg, .png, etc.)
        matches = glob.glob(os.path.join(src_dir, f"{img_id}.*"))
        if matches:
            for src in matches:
                shutil.copy2(src, dst_dir)
            copied += 1
        else:
            missing.append(img_id)
    return copied, missing

# --- Calibration FNs ---
copied1, missing1 = copy_images(fp_all_test1, data_dir, out_test1)
print(f"[test1] Copied: {copied1} | Not found: {len(missing1)}")
if missing1:
    print(f"  Missing IDs: {missing1}")

# --- Validation FNs ---
copied2, missing2 = copy_images(fp_all_test2, data_dir, out_test2)
print(f"[test2] Copied: {copied2} | Not found: {len(missing2)}")
if missing2:
    print(f"  Missing IDs: {missing2}")

[test1] Copied: 10 | Not found: 0
[test2] Copied: 6 | Not found: 0


### TP

In [17]:
# ===========================================================
# FN IMAGES MISSED BY ALL MODELS (calibration & validation)
# ===========================================================

# --- Calibration (test1) ---
tp_sets_test1 = [set(ids) for ids in TP_test1.values()]
tp_all_test1  = set.intersection(*tp_sets_test1)
tp_unique_test1 = sorted(set.union(*tp_sets_test1))

print(f"Calibration (test1): {len(tp_all_test1)} images are TP in ALL models")
print(f"Unique TP image IDs (any model): {tp_unique_test1}")
print(tp_all_test1)

# --- Validation (test2) ---
tp_sets_test2 = [set(ids) for ids in TP_test2.values()]
tp_all_test2  = set.intersection(*tp_sets_test2)
tp_unique_test2 = sorted(set.union(*tp_sets_test2))

print(f"\nValidation  (test2): {len(tp_all_test2)} images are TP in ALL models")
print(f"Unique TP image IDs (any model): {tp_unique_test2}")
print(tp_all_test2)

Calibration (test1): 10 images are TP in ALL models
Unique TP image IDs (any model): [610004, 610298, 618209, 625049, 625050, 625052, 625154, 625155, 630291, 630292, 632008, 643544, 643545, 645041]
{618209, 625155, 632008, 625050, 645041, 630291, 610004, 630292, 643544, 610298}

Validation  (test2): 12 images are TP in ALL models
Unique TP image IDs (any model): [608376, 608377, 610005, 613277, 618210, 623793, 625051, 627174, 629264, 629265, 644557, 645040, 648736, 648737]
{648736, 648737, 627174, 644557, 629264, 623793, 629265, 645040, 610005, 608376, 608377, 613277}


In [18]:
import shutil

data_dir   = "/projects/retprogression/test_dataset_10032025"
out_test1  = "/projects/retprogression/Gradable_allWrong_TP_test1"
out_test2  = "/projects/retprogression/Gradable_allWrong_TP_test2"

os.makedirs(out_test1, exist_ok=True)
os.makedirs(out_test2, exist_ok=True)

def copy_images(image_ids, src_dir, dst_dir):
    copied, missing = 0, []
    for img_id in image_ids:
        # Glob in case the extension varies (.jpg, .png, etc.)
        matches = glob.glob(os.path.join(src_dir, f"{img_id}.*"))
        if matches:
            for src in matches:
                shutil.copy2(src, dst_dir)
            copied += 1
        else:
            missing.append(img_id)
    return copied, missing

# --- Calibration FNs ---
copied1, missing1 = copy_images(tp_all_test1, data_dir, out_test1)
print(f"[test1] Copied: {copied1} | Not found: {len(missing1)}")
if missing1:
    print(f"  Missing IDs: {missing1}")

# --- Validation FNs ---
copied2, missing2 = copy_images(tp_all_test2, data_dir, out_test2)
print(f"[test2] Copied: {copied2} | Not found: {len(missing2)}")
if missing2:
    print(f"  Missing IDs: {missing2}")

[test1] Copied: 10 | Not found: 0
[test2] Copied: 12 | Not found: 0


### TN

In [19]:
# ===========================================================
# FN IMAGES MISSED BY ALL MODELS (calibration & validation)
# ===========================================================

# --- Calibration (test1) ---
tn_sets_test1 = [set(ids) for ids in TN_test1.values()]
tn_all_test1  = set.intersection(*tn_sets_test1)
tn_unique_test1 = sorted(set.union(*tn_sets_test1))

print(f"Calibration (test1): {len(tn_all_test1)} images are TN in ALL models")
print(f"Unique TN image IDs (any model): {tn_unique_test1}")
print(tn_all_test1)

# --- Validation (test2) ---
tn_sets_test2 = [set(ids) for ids in TN_test2.values()]
tn_all_test2  = set.intersection(*tn_sets_test2)
tn_unique_test2 = sorted(set.union(*tn_sets_test2))

print(f"\nValidation  (test2): {len(tn_all_test2)} images are TN in ALL models")
print(f"Unique TN image IDs (any model): {tn_unique_test2}")
print(tn_all_test2)

Calibration (test1): 771 images are TN in ALL models
Unique TN image IDs (any model): [605394, 605395, 605397, 605539, 605630, 605639, 605684, 605687, 605756, 605933, 605934, 605967, 605982, 605983, 605984, 606305, 606307, 606371, 606381, 606471, 606472, 606504, 606521, 606736, 606818, 606819, 607967, 608042, 608043, 608103, 608148, 608150, 608155, 608173, 608180, 608338, 608343, 608345, 608346, 608380, 608381, 608547, 608672, 608673, 608680, 608716, 608729, 608926, 608970, 609021, 609356, 609681, 609685, 609718, 609753, 609777, 609778, 609856, 609895, 609896, 609920, 609921, 609922, 609951, 609952, 609968, 609969, 610204, 610205, 610214, 610379, 610380, 610474, 610475, 610496, 610519, 610629, 610630, 610696, 610731, 610732, 610739, 610746, 610777, 610821, 610990, 611002, 611003, 611096, 611099, 611158, 611159, 611163, 611220, 611345, 611346, 611384, 611387, 611405, 611504, 611507, 611586, 611598, 611599, 611601, 611685, 611724, 611996, 611998, 612004, 612020, 612022, 612023, 612063, 6

In [20]:
import shutil

data_dir   = "/projects/retprogression/test_dataset_10032025"
out_test1  = "/projects/retprogression/Gradable_allWrong_TN_test1"
out_test2  = "/projects/retprogression/Gradable_allWrong_TN_test2"

os.makedirs(out_test1, exist_ok=True)
os.makedirs(out_test2, exist_ok=True)

def copy_images(image_ids, src_dir, dst_dir):
    copied, missing = 0, []
    for img_id in image_ids:
        # Glob in case the extension varies (.jpg, .png, etc.)
        matches = glob.glob(os.path.join(src_dir, f"{img_id}.*"))
        if matches:
            for src in matches:
                shutil.copy2(src, dst_dir)
            copied += 1
        else:
            missing.append(img_id)
    return copied, missing

# --- Calibration FNs ---
copied1, missing1 = copy_images(tn_all_test1, data_dir, out_test1)
print(f"[test1] Copied: {copied1} | Not found: {len(missing1)}")
if missing1:
    print(f"  Missing IDs: {missing1}")

# --- Validation FNs ---
copied2, missing2 = copy_images(tn_all_test2, data_dir, out_test2)
print(f"[test2] Copied: {copied2} | Not found: {len(missing2)}")
if missing2:
    print(f"  Missing IDs: {missing2}")

[test1] Copied: 771 | Not found: 0
[test2] Copied: 784 | Not found: 0
